In [0]:
%run ../00-common/01.enviroment-config

In [0]:

bronze_table=f"{catalog_name}.{bronze_schema}.results"
silver_table=f"{catalog_name}.{silver_schema}.results"

In [0]:
from pyspark.sql import functions as F

In [0]:
results_df=(
    spark.table(bronze_table)
    .select("season",
                      "round","constructorId","driverId","date","raceName","grid","laps","number","points","position","positionText","status","ingestion_timestamp","source_file")
    .withColumnsRenamed({
        "constructorId":"constructor_id",
        "driverId":"driver_id",
        "raceName":"race_name",
        "date":"race_date",
        "grid":"grid_position",
        "laps": "completed_laps",
        "number":"car_number",
        "position":"final_position",
        "positionText":"final_position_text"}))
     

In [0]:
#for season,round,constructor_id,driver_id

results_valid_df=(results_df
                  .filter(F.col("season").isNotNull() & 
                          F.col("driver_id").isNotNull() &  
                          F.col("constructor_id").isNotNull() & 
                          F.col("round").isNotNull() 
                  )
                  .dropDuplicates(["season","round","constructor_id","driver_id"])
                  )

In [0]:
display(results_df.count()-results_valid_df.count())


In [0]:
#trasform values of race name to title case
results_final_df=results_valid_df.withColumn("race_name",F.initcap(F.col("race_name")))

In [0]:
(
    results_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))